# Projet Machine Learning - Students Performance
Dataset : StudentsPerformance.csv

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, LogisticRegression
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree, export_text
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, AdaBoostClassifier, GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, auc, mean_squared_error, r2_score, mean_absolute_error,
    classification_report, silhouette_score
)
from scipy.cluster.hierarchy import dendrogram, linkage

SEED = 42
plt.rcParams['figure.dpi'] = 100

## 2. Chargement et exploration des donnees

In [ ]:
df = pd.read_csv('StudentsPerformance.csv')
print('Shape:', df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
print('Valeurs manquantes:')
print(df.isnull().sum())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, ['math score', 'reading score', 'writing score']):
    ax.hist(df[col], bins=20, edgecolor='white')
    ax.set_title(col)
    ax.set_xlabel('Score')
plt.suptitle('Distribution des scores')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
cats = [('gender', 'Genre'), ('lunch', 'Lunch'), ('test preparation course', 'Test Prep'), ('race/ethnicity', 'Race')]
for ax, (col, title) in zip(axes.flatten(), cats):
    df.groupby(col)[['math score','reading score','writing score']].mean().plot(kind='bar', ax=ax)
    ax.set_title(title)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=30)
plt.suptitle('Scores moyens par variable categorielle')
plt.tight_layout()
plt.show()

In [ ]:
corr = df[['math score', 'reading score', 'writing score']].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Matrice de correlation')
plt.show()

## 3. Preparation des donnees

In [ ]:
df['avg_score'] = (df['math score'] + df['reading score'] + df['writing score']) / 3
df['pass_fail'] = (df['avg_score'] >= 60).astype(int)

edu_order = {
    'some high school': 0, 'high school': 1, 'some college': 2,
    "associate's degree": 3, "bachelor's degree": 4, "master's degree": 5
}
df['parental_edu_num'] = df['parental level of education'].map(edu_order)
df['gender_num']   = (df['gender'] == 'female').astype(int)
df['lunch_num']    = (df['lunch'] == 'standard').astype(int)
df['testprep_num'] = (df['test preparation course'] == 'completed').astype(int)
df = pd.get_dummies(df, columns=['race/ethnicity'], prefix='race')

FEATURES = ['gender_num', 'lunch_num', 'testprep_num', 'parental_edu_num',
            'race_group A', 'race_group B', 'race_group C', 'race_group D', 'race_group E']

X = df[FEATURES].astype(float)
y_reg = df['math score'].astype(float)
y_clf = df['pass_fail'].astype(int)

X_train, X_test, y_train_r, y_test_r = train_test_split(X, y_reg, test_size=0.2, random_state=SEED)
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X, y_clf, test_size=0.2, random_state=SEED, stratify=y_clf)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print('Train:', X_train.shape, '| Test:', X_test.shape)
print('Pass rate train:', round(y_train_c.mean(), 3))

## 4. Modeles Lineaires

In [ ]:
models_reg = {
    'OLS':        LinearRegression(),
    'Ridge':      Ridge(alpha=1.0),
    'Lasso':      Lasso(alpha=0.1),
    'ElasticNet': ElasticNet(alpha=0.1, l1_ratio=0.5)
}

results_reg = {}
for name, m in models_reg.items():
    m.fit(X_train_s, y_train_r)
    p = m.predict(X_test_s)
    results_reg[name] = {
        'RMSE': round(np.sqrt(mean_squared_error(y_test_r, p)), 4),
        'MAE':  round(mean_absolute_error(y_test_r, p), 4),
        'R2':   round(r2_score(y_test_r, p), 4)
    }

pd.DataFrame(results_reg).T

In [ ]:
ridge = Ridge(alpha=1.0).fit(X_train_s, y_train_r)
lasso = Lasso(alpha=0.1).fit(X_train_s, y_train_r)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, m, title in zip(axes, [ridge, lasso], ['Ridge - coefficients', 'Lasso - coefficients']):
    ax.bar(FEATURES, m.coef_)
    ax.set_title(title)
    ax.set_xticklabels(FEATURES, rotation=45, ha='right', fontsize=8)
    ax.axhline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=SEED)
lr.fit(X_train_s, y_train_c)
y_pred_lr = lr.predict(X_test_s)

print(classification_report(y_test_c, y_pred_lr, target_names=['Fail', 'Pass']))

ConfusionMatrixDisplay.from_predictions(y_test_c, y_pred_lr, display_labels=['Fail','Pass'], cmap='Blues')
plt.title('Confusion Matrix - Logistic Regression')
plt.show()

## 5. Arbres de Decision

In [ ]:
dt = DecisionTreeClassifier(criterion='gini', max_depth=4, min_samples_leaf=10, random_state=SEED)
dt.fit(X_train_c, y_train_c)
y_pred_dt = dt.predict(X_test_c)

print('Accuracy:', round(accuracy_score(y_test_c, y_pred_dt), 4))
print('F1-Score:', round(f1_score(y_test_c, y_pred_dt), 4))

In [ ]:
plt.figure(figsize=(20, 7))
plot_tree(dt, feature_names=FEATURES, class_names=['Fail','Pass'], filled=True, rounded=True, fontsize=8)
plt.title('Arbre de Decision (max_depth=4, Gini)')
plt.show()

In [ ]:
train_scores, test_scores = [], []
depths = range(1, 20)
for d in depths:
    m = DecisionTreeClassifier(max_depth=d, random_state=SEED)
    m.fit(X_train_c, y_train_c)
    train_scores.append(accuracy_score(y_train_c, m.predict(X_train_c)))
    test_scores.append(accuracy_score(y_test_c, m.predict(X_test_c)))

plt.plot(depths, train_scores, label='Train')
plt.plot(depths, test_scores, label='Test')
plt.axvline(4, color='gray', linestyle='--', label='depth=4')
plt.xlabel('max_depth')
plt.ylabel('Accuracy')
plt.title('Biais-Variance - Decision Tree')
plt.legend()
plt.show()

In [ ]:
fi = pd.Series(dt.feature_importances_, index=FEATURES).sort_values(ascending=False)
fi.plot(kind='bar')
plt.title('Feature Importance - Decision Tree')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 6. KNN

In [ ]:
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
f1_scores_k = []
for k in range(1, 31):
    knn = KNeighborsClassifier(n_neighbors=k)
    f1_scores_k.append(cross_val_score(knn, X_train_s, y_train_c, cv=cv, scoring='f1').mean())

best_k = np.argmax(f1_scores_k) + 1
plt.plot(range(1, 31), f1_scores_k, 'o-')
plt.axvline(best_k, color='red', linestyle='--', label=f'K={best_k}')
plt.xlabel('K')
plt.ylabel('F1 (CV 5-fold)')
plt.title('Selection du K optimal')
plt.legend()
plt.show()
print('Meilleur K:', best_k)

In [ ]:
knn = KNeighborsClassifier(n_neighbors=best_k, weights='distance')
knn.fit(X_train_s, y_train_c)
y_pred_knn = knn.predict(X_test_s)

print(classification_report(y_test_c, y_pred_knn, target_names=['Fail','Pass']))

ConfusionMatrixDisplay.from_predictions(y_test_c, y_pred_knn, display_labels=['Fail','Pass'], cmap='Greens')
plt.title(f'Confusion Matrix - KNN (K={best_k})')
plt.show()

In [ ]:
dist_results = {}
for metric in ['euclidean', 'manhattan', 'chebyshev']:
    m = KNeighborsClassifier(n_neighbors=best_k, metric=metric)
    m.fit(X_train_s, y_train_c)
    p = m.predict(X_test_s)
    dist_results[metric] = {'Accuracy': round(accuracy_score(y_test_c, p), 4),
                             'F1': round(f1_score(y_test_c, p), 4)}
pd.DataFrame(dist_results).T

## 7. Ensemble Learning

In [ ]:
ensemble_models = {
    'Random Forest':      RandomForestClassifier(n_estimators=100, max_depth=6, random_state=SEED),
    'AdaBoost':           AdaBoostClassifier(n_estimators=100, learning_rate=0.5, random_state=SEED),
    'Gradient Boosting':  GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=SEED)
}

ens_results = {}
trained = {}
for name, m in ensemble_models.items():
    m.fit(X_train_c, y_train_c)
    p = m.predict(X_test_c)
    trained[name] = m
    ens_results[name] = {
        'Accuracy':  round(accuracy_score(y_test_c, p), 4),
        'Precision': round(precision_score(y_test_c, p), 4),
        'Recall':    round(recall_score(y_test_c, p), 4),
        'F1':        round(f1_score(y_test_c, p), 4)
    }

pd.DataFrame(ens_results).T

In [ ]:
rf = trained['Random Forest']
fi_rf = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=False)
fi_rf.plot(kind='bar')
plt.title('Feature Importance - Random Forest')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 8. Methodes Rule-Based

In [ ]:
dt_rules = DecisionTreeClassifier(criterion='gini', max_depth=3, min_samples_leaf=15, random_state=SEED)
dt_rules.fit(X_train_c, y_train_c)
print(export_text(dt_rules, feature_names=FEATURES))

In [ ]:
N = len(df)
rules = [
    ('IF lunch=standard => Pass',                             df['lunch_num']==1,                                     df['pass_fail']==1),
    ('IF testprep=done AND lunch=standard => Pass',           (df['testprep_num']==1)&(df['lunch_num']==1),           df['pass_fail']==1),
    ('IF lunch=free/reduced AND parental_edu<=1 => Fail',    (df['lunch_num']==0)&(df['parental_edu_num']<=1),       df['pass_fail']==0),
]

print(f'{"Regle":<55} | Coverage | Accuracy')
print('-'*80)
for rule, ant, con in rules:
    cov = ant.sum() / N
    acc = (ant & con).sum() / ant.sum() if ant.sum() > 0 else 0
    print(f'{rule:<55} | {cov:.2%}   | {acc:.2%}')

## 9. Clustering (Non Supervise)

In [ ]:
X_viz = df[['math score', 'avg_score']].values
X_viz_s = StandardScaler().fit_transform(X_viz)

In [ ]:
sse = [KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(X_viz_s).inertia_ for k in range(1, 11)]
plt.plot(range(1, 11), sse, 'o-')
plt.axvline(3, color='red', linestyle='--', label='K=3')
plt.xlabel('K')
plt.ylabel('SSE')
plt.title('Elbow Method')
plt.legend()
plt.show()

In [ ]:
km = KMeans(n_clusters=3, n_init=10, random_state=SEED)
labels_km = km.fit_predict(X_viz_s)
print('Silhouette Score:', round(silhouette_score(X_viz_s, labels_km), 4))

plt.figure(figsize=(7, 5))
for i in range(3):
    mask = labels_km == i
    plt.scatter(X_viz[mask, 0], X_viz[mask, 1], label=f'Cluster {i+1}', alpha=0.6, s=20)
plt.xlabel('Math Score')
plt.ylabel('Avg Score')
plt.title('K-Means (K=3)')
plt.legend()
plt.show()

In [ ]:
idx = np.random.RandomState(SEED).choice(len(X_viz_s), 80, replace=False)
Z = linkage(X_viz_s[idx], method='ward')
dendrogram(Z, leaf_font_size=6)
plt.title('Dendrogramme - Ward Linkage')
plt.xlabel('Etudiants')
plt.ylabel('Distance')
plt.show()

print('Silhouette par methode de linkage:')
for method in ['ward', 'complete', 'average', 'single']:
    lbl = AgglomerativeClustering(n_clusters=3, linkage=method).fit_predict(X_viz_s)
    print(f'  {method}: {round(silhouette_score(X_viz_s, lbl), 4)}')

In [ ]:
db = DBSCAN(eps=0.5, min_samples=10)
labels_db = db.fit_predict(X_viz_s)
n_clusters = len(set(labels_db)) - (1 if -1 in labels_db else 0)
n_noise    = (labels_db == -1).sum()
print(f'Clusters: {n_clusters} | Bruit (outliers): {n_noise}')

plt.figure(figsize=(7, 5))
for label in sorted(set(labels_db)):
    mask = labels_db == label
    lbl = 'Bruit' if label == -1 else f'Cluster {label+1}'
    plt.scatter(X_viz[mask, 0], X_viz[mask, 1], label=lbl, s=20,
                c='black' if label==-1 else None, alpha=0.6)
plt.xlabel('Math Score')
plt.ylabel('Avg Score')
plt.title('DBSCAN (eps=0.5, min_samples=10)')
plt.legend()
plt.show()

## 10. Evaluation et Comparaison des modeles

In [ ]:
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

all_clf = {
    'Logistic Regression': Pipeline([('sc', StandardScaler()), ('clf', LogisticRegression(max_iter=1000))]),
    'Decision Tree':       DecisionTreeClassifier(max_depth=4, random_state=SEED),
    'KNN':                 Pipeline([('sc', StandardScaler()), ('clf', KNeighborsClassifier(n_neighbors=best_k))]),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=SEED),
    'AdaBoost':            AdaBoostClassifier(n_estimators=100, random_state=SEED),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=100, random_state=SEED),
}

cv_res = {}
for name, m in all_clf.items():
    cv_res[name] = {
        'Accuracy':  round(cross_val_score(m, X, y_clf, cv=cv5, scoring='accuracy').mean(), 4),
        'Precision': round(cross_val_score(m, X, y_clf, cv=cv5, scoring='precision').mean(), 4),
        'Recall':    round(cross_val_score(m, X, y_clf, cv=cv5, scoring='recall').mean(), 4),
        'F1':        round(cross_val_score(m, X, y_clf, cv=cv5, scoring='f1').mean(), 4),
    }

df_cv = pd.DataFrame(cv_res).T.sort_values('F1', ascending=False)
df_cv

In [ ]:
plt.figure(figsize=(9, 6))
plt.plot([0,1],[0,1],'k--', label='Aleatoire (AUC=0.5)')

for name, m in all_clf.items():
    m.fit(X_train_c, y_train_c)
    prob = m.predict_proba(X_test_c)[:, 1]
    fpr, tpr, _ = roc_curve(y_test_c, prob)
    plt.plot(fpr, tpr, label=f'{name} (AUC={round(auc(fpr,tpr),3)})')

plt.xlabel('FPR')
plt.ylabel('TPR')
plt.title('Courbes ROC')
plt.legend(fontsize=8, loc='lower right')
plt.show()

In [ ]:
all_reg = {
    'Ridge':           Pipeline([('sc', StandardScaler()), ('r', Ridge())]),
    'Lasso':           Pipeline([('sc', StandardScaler()), ('r', Lasso(alpha=0.1))]),
    'ElasticNet':      Pipeline([('sc', StandardScaler()), ('r', ElasticNet(alpha=0.1))]),
    'Decision Tree':   DecisionTreeRegressor(max_depth=5, random_state=SEED),
    'Random Forest':   RandomForestRegressor(n_estimators=100, random_state=SEED),
    'Grad. Boosting':  GradientBoostingRegressor(n_estimators=100, random_state=SEED),
    'KNN':             Pipeline([('sc', StandardScaler()), ('r', KNeighborsRegressor(n_neighbors=best_k))]),
}

reg_res = {}
for name, m in all_reg.items():
    m.fit(X_train, y_train_r)
    p = m.predict(X_test)
    reg_res[name] = {
        'RMSE': round(np.sqrt(mean_squared_error(y_test_r, p)), 4),
        'MAE':  round(mean_absolute_error(y_test_r, p), 4),
        'R2':   round(r2_score(y_test_r, p), 4)
    }

pd.DataFrame(reg_res).T.sort_values('R2', ascending=False)

In [ ]:
best_clf_name = df_cv['F1'].idxmax()
best_clf_f1   = df_cv['F1'].max()

df_reg_final = pd.DataFrame(reg_res).T.sort_values('R2', ascending=False)
best_reg_name = df_reg_final['R2'].idxmax()
best_reg_r2   = df_reg_final['R2'].max()

print('===== Resume Final =====')
print(f'Meilleur classificateur : {best_clf_name} (F1={best_clf_f1})')
print(f'Meilleur regresseur     : {best_reg_name} (R2={best_reg_r2})')
print(f'K-Means Silhouette      : {round(silhouette_score(X_viz_s, labels_km), 4)}')